Example to showcase how to use vLLM + ReActTrain Agent + Qwen3-4B-Instruct + MCP-U's RolloutEngine.
You need to first deploy Qwen3-4B-Instruct via vLLM:
```
MODEL=Qwen3-4B-Instruct-2507
API_KEY="token-abc123"
PORT=2024


python -m vllm.entrypoints.openai.api_server \
    --model $MODEL \
    --served-model-name $MODEL \
    --tensor-parallel-size 1 \
    --async-scheduling \
    --gpu-memory-utilization 0.95 \
    --api-key $API_KEY \
    --host "0.0.0.0" \
    --port $PORT \
    --disable-log-requests
```
Then you can go through the following examples.

In [ ]:
import asyncio
from mcpuniverse.rl import RolloutEngine, RolloutConfig

In [2]:
async def example_basic_react_train():
    """
    Example 1: Basic ReActTrain Agent query

    ReActTrain Agent uses ChatML format (<|im_start|>...<|im_end|>) for tool calls.
    Output format is JSON: {"thought": ..., "action": {...}} or {"thought": ..., "answer": ...}
    """
    print("\n" + "=" * 60)
    print("Example 1: Basic ReActTrain Agent Query (Qwen3)")
    print("=" * 60)

    config = RolloutConfig.from_dict({
        "llm_type": "vllm_local",
        "llm_config": {
            "model_name": "Qwen3-4B-Instruct-2507",
            "base_url": "http://localhost:2024",
            "api_key": "token-abc123",
            "temperature": 0.7,
            "max_completion_tokens": 4096,
            "reasoning": "low",
        },
        "agent_mode": "react_train",
        "formatter_type": "qwen3",
        "mcp_servers": [{"name": "weather"}],
        "generator": {
            "num_trajectories": 3,
            "max_iterations": 10,
        },
    })

    engine = RolloutEngine(config)

    output = await engine.run([
        {
            "instruction": "What's the weather in New York City?",
            "instance_id": "nyc_weather",
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "temperature"},
            ],
        }
    ])

    print(f"\nFinal response: {output.responses[0]}")
    print(f"Reward: {output.rewards[0]}")
    print(f"Finish reason: {output.finish_reasons[0]}")

    if output.trajectories:
        traj = output.trajectories[0]
        print(f"\nSteps: {traj['num_steps']}")
        print(f"Tool calls: {traj['num_tool_calls']}")
        print(f"Running time: {traj['running_time']:.2f}s")

        print("\n--- Full trace text (full_trace_text) ---")
        full_text = traj.get("full_trace_text", "")
        print(full_text)

In [3]:
from transformers import AutoTokenizer
from mcpuniverse.rl.formatters import get_formatter

async def example_multi_step_reasoning():
    """
    Example 2: Multi-step reasoning

    ReActTrain Agent decomposes complex problems and calls tools multiple times.
    """
    print("\n" + "=" * 60)
    print("Example 2: Multi-step Reasoning (Qwen3)")
    print("=" * 60)

    config = RolloutConfig.from_dict({
        "llm_type": "vllm_local",
        "llm_config": {
            "model_name": "Qwen3-4B-Instruct-2507",
            "base_url": "http://localhost:2024",
            "api_key": "token-abc123",
            "reasoning": "low",
        },
        "agent_mode": "react_train",
        "formatter_type": "qwen3",
        "mcp_servers": [{"name": "weather"}],
        "generator": {
            "num_trajectories": 1,
            "max_iterations": 15,
        },
    })

    engine = RolloutEngine(config)

    output = await engine.run([
        {
            "instruction": "Compare the weather between NYC and Los Angeles. "
                          "Which city is warmer right now? You need to use the weather tool!",
        }
    ])

    print(f"\nFinal response: {output.responses[0]}")

    if output.trajectories:
        traj = output.trajectories[0]
        print(f"\nTotal steps: {traj['num_steps']}")
        print(f"Tool calls: {traj['num_tool_calls']}")
        print(f"Running time: {traj['running_time']:.2f}s")

        # Tokenize with mask test
        print("\n" + "=" * 60)
        print("Tokenize with Mask Test (Qwen3 Formatter)")
        print("=" * 60)

        tokenizer = AutoTokenizer.from_pretrained(
            "Qwen3-4B-Instruct-2507",
            trust_remote_code=True,
        )

        formatter = get_formatter("qwen3")
        full_trace_text = traj.get("full_trace_text", "")

        if full_trace_text:
            formatter_output = formatter.format_trace(
                full_trace_text,
                "Compare the weather between NYC and Los Angeles...",
            )

            prompt_tokens, output_tokens, loss_mask = formatter.tokenize_with_mask(
                formatter_output, tokenizer,
            )

            print(f"\n{'=' * 60}")
            print("[Prompt / Input] - Not trained")
            print(f"{'=' * 60}")
            print(f"Token count: {len(prompt_tokens)}")
            print(f"\n--- Prompt text ---")
            print(formatter_output.prompt_text)
            print(f"--- End prompt ---")

            print(f"\n{'=' * 60}")
            print("[Output] - Trained according to mask")
            print(f"{'=' * 60}")
            print(f"Token count: {len(output_tokens)}")
            print(f"\n--- Output text ---")
            print(formatter_output.output_text)
            print(f"--- End output ---")

            print(f"\n{'=' * 60}")
            print("[Loss Mask Stats]")
            print(f"{'=' * 60}")
            print(f"Total length: {len(loss_mask)}")
            print(f"Trainable tokens (mask=1): {sum(loss_mask)}")
            print(f"Non-trainable tokens (mask=0): {len(loss_mask) - sum(loss_mask)}")
            trainable_ratio = sum(loss_mask) / len(loss_mask) * 100 if loss_mask else 0
            print(f"Trainable ratio: {trainable_ratio:.1f}%")

            print(f"\n{'=' * 60}")
            print("[Segment Details]")
            print(f"{'=' * 60}")
            for i, seg in enumerate(formatter_output.output_segments):
                raw = seg.get("raw", "") or seg.get("content", "")
                trainable = seg.get("trainable", False)
                role = seg.get("role", "")
                channel = seg.get("channel", "")
                raw_preview = raw[:150] + "..." if len(raw) > 150 else raw
                print(f"\n  Segment {i}: role={role}, channel={channel}")
                print(f"    Trainable: {'mask=1' if trainable else 'mask=0'}")
                print(f"    Length: {len(raw)} chars")
                print(f"    Preview: {raw_preview}")


In [ ]:
async def example_batch_processing():
    """
    Example 3: Batch processing with evaluators
    """
    print("\n" + "=" * 60)
    print("Example 3: Batch Processing with Evaluators")
    print("=" * 60)

    config = RolloutConfig.from_dict({
        "llm_type": "vllm_local",
        "llm_config": {
            "model_name": "Qwen3-4B-Instruct-2507",
            "base_url": "http://localhost:2024",
            "api_key": "token-abc123",
            "temperature": 0.7,
        },
        "agent_mode": "react_train",
        "use_sample_servers": True,
        "generator": {
            "num_trajectories": 4,
            "max_iterations": 10,
        },
        "dispatcher": {
            "type": "async_pipeline",
            "max_parallel_agents": 32,
        },
    })

    engine = RolloutEngine(config)

    batch = [
        {
            "instruction": "Compare the weather between New York and Los Angeles. Which city is warmer right now?",
            "instance_id": "nyc_vs_la",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "York"},
                {"func": "raw", "op": "contain", "value": "Angeles"},
            ],
        },
        {
            "instruction": "If current weather in Los Angeles is sunny, find a cafe in Los Angeles. "
                          "If current weather in Los Angeles is rainy, find a park in Los Angeles.",
            "instance_id": "la",
            "mcp_servers": [{"name": "google-maps"}, {"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Los Angeles"},
            ],
        },
        {
            "instruction": "What's the weather in San Francisco?",
            "instance_id": "sf",
            "mcp_servers": [{"name": "weather"}],
            "evaluators": [
                {"func": "raw", "op": "contain", "value": "Francisco"},
            ],
        },
    ]

    output = await engine.run(batch)

    num_traj = config.generator.num_trajectories
    print(f"\nProcessed {len(batch)} instances, {len(output.responses)} trajectories ({num_traj} per instance)")
    print(f"Success rate: {output.rollout_metrics.get('rollout_metrics/success_rate', 0):.2%}")

    for i, traj in enumerate(output.trajectories):
        instance_id = traj.get("instance_id", i)
        traj_id = traj.get("trajectory_id", 0)
        resp = traj.get("response", "")
        reward = traj.get("reward", 0.0)
        print(f"\nInstance {instance_id} - Trajectory {traj_id}:")
        print(f"  Response: {resp[:200]}..." if len(resp) > 200 else f"  Response: {resp}")
        print(f"  Reward: {reward}")